## Run in root directory rather than shared/ for reduced s3 spend and better performance

Import necessary libraries

In [20]:
import os
import pandas as pd

import sagemaker
from sagemaker.workflow.execution_variables import ExecutionVariables
from sagemaker.workflow.function_step import step
from sagemaker.workflow.parameters import ParameterString
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.fail_step import FailStep

Declare some variables used later

### Change these constants:

In [21]:
FEAT_DATA_PATH = "s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/data/training/planetary_computer/marsabit_features_full.parquet"
TARGET_DATA_PATH = "s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/data/training/ibli/target_data_pipeline.csv"
DATE_JOIN_KEY = ['year', 'month']
LOCATION_JOIN_KEY = 'hhid'

LABEL_COLUMN = 'tlu_loss_ratio'
FEAT_LS = [
    'soil',
    'ppt',
    'pdsi',
    'vpd',
    'soil_anom',
    'ppt_anom',
    'pdsi_anom',
    'vpd_anom',
    'drought_mild_tc',
    'drought_severe_tc',
    'heat_stress',
    'compound_stress_tc',
    'soil_delta_3m',
    'drought_duration_pdsi',
    'months_since_pos_ppt_anom',
    'is_lrld_tc',
    'pdsi_anom_x_lrld',
    'compound_x_lrld',
    'ppt_lag1',
    'ppt_lag2',
    'ppt_lag3',
    'vpd_lag1',
    'vpd_lag2',
    'vpd_lag3',
    'ppt_anom_lag1',
    'ppt_anom_lag2',
    'ppt_anom_lag3',
    'vpd_anom_lag1',
    'vpd_anom_lag2',
    'vpd_anom_lag3',
    'compound_stress_lag1',
    'compound_stress_lag2',
    'compound_stress_lag3',
    'heat_stress_lag1',
    'heat_stress_lag2',
    'heat_stress_lag3',
    'soil_lag1',
    'soil_lag2',
    'soil_lag3',
    'soil_lag4',
    'soil_lag5',
    'soil_lag6',
    'pdsi_lag1',
    'pdsi_lag2',
    'pdsi_lag3',
    'pdsi_lag4',
    'pdsi_lag5',
    'pdsi_lag6',
    'soil_anom_lag1',
    'soil_anom_lag2',
    'soil_anom_lag3',
    'soil_anom_lag4',
    'soil_anom_lag5',
    'soil_anom_lag6',
    'pdsi_anom_lag1',
    'pdsi_anom_lag2',
    'pdsi_anom_lag3',
    'pdsi_anom_lag4',
    'pdsi_anom_lag5',
    'pdsi_anom_lag6',
    'drought_mild_lag1',
    'drought_severe_lag1',
    'ppt_anom_roll3',
    'ppt_anom_roll6',
    'soil_anom_roll3',
    'pdsi_anom_roll6',
    'vpd_anom_roll3',
    'ndvi_250m',
    'evi_250m',
    'lai',
    'fpar',
    'lst_day',
    'lst_night',
    'et',
    'pet',
    'sr_red',
    'sr_nir',
    'sr_swir1',
    'sr_swir2',
    'fire_mask',
    'fire_frp_max',
    'fire_frp_sum',
    's1_vv',
    's1_vh',
    's2_red',
    's2_nir',
    's2_swir1',
    'dem',
    'jrc_occurrence',
    'jrc_seasonality',
    'wc_trees',
    'wc_shrubland',
    'wc_grassland',
    'wc_cropland',
    'wc_builtup',
    'wc_bare',
    'wc_water',
    'wc_wetland',
    'dem_std',
    'dem_range',
    'evi2',
    'savi',
    'msavi',
    'osavi',
    'ndwi',
    'lswi',
    'nbr',
    'bsi',
    'swir_ratio',
    's2_ndvi',
    's2_ndwi',
    'sar_vv_vh_ratio',
    'sar_rvi',
    'et_deficit',
    'et_fraction',
    'lst_diurnal_range',
    'is_lrld_pc',
    'is_mam',
    'is_ond',
    'ndvi_250m_lag1',
    'ndvi_250m_lag2',
    'ndvi_250m_lag3',
    'ndvi_250m_diff1',
    'ndvi_250m_ratio1',
    'ndvi_250m_yoy_diff',
    'ndvi_250m_yoy_ratio',
    'ndvi_250m_roll3_mean',
    'ndvi_250m_roll3_std',
    'ndvi_250m_roll3_sum',
    'evi_250m_lag1',
    'evi_250m_lag2',
    'evi_250m_lag3',
    'evi_250m_diff1',
    'evi_250m_ratio1',
    'evi_250m_yoy_diff',
    'evi_250m_yoy_ratio',
    'evi_250m_roll3_mean',
    'evi_250m_roll3_std',
    'evi_250m_roll3_sum',
    'lst_day_lag1',
    'lst_day_lag2',
    'lst_day_lag3',
    'lst_day_diff1',
    'lst_day_ratio1',
    'lst_day_yoy_diff',
    'lst_day_yoy_ratio',
    'lst_day_roll3_mean',
    'lst_day_roll3_std',
    'lst_day_roll3_sum',
    'lst_night_lag1',
    'lst_night_lag2',
    'lst_night_lag3',
    'lst_night_diff1',
    'lst_night_ratio1',
    'lst_night_yoy_diff',
    'lst_night_yoy_ratio',
    'lst_night_roll3_mean',
    'lst_night_roll3_std',
    'lst_night_roll3_sum',
    'lai_lag1',
    'lai_lag2',
    'lai_lag3',
    'lai_diff1',
    'lai_ratio1',
    'lai_yoy_diff',
    'lai_yoy_ratio',
    'lai_roll3_mean',
    'lai_roll3_std',
    'lai_roll3_sum',
    'fpar_lag1',
    'fpar_lag2',
    'fpar_lag3',
    'fpar_diff1',
    'fpar_ratio1',
    'fpar_yoy_diff',
    'fpar_yoy_ratio',
    'fpar_roll3_mean',
    'fpar_roll3_std',
    'fpar_roll3_sum',
    'sr_nir_lag1',
    'sr_nir_lag2',
    'sr_nir_lag3',
    'sr_nir_diff1',
    'sr_nir_ratio1',
    'sr_nir_yoy_diff',
    'sr_nir_yoy_ratio',
    'sr_nir_roll3_mean',
    'sr_nir_roll3_std',
    'sr_nir_roll3_sum',
    'sr_red_lag1',
    'sr_red_lag2',
    'sr_red_lag3',
    'sr_red_diff1',
    'sr_red_ratio1',
    'sr_red_yoy_diff',
    'sr_red_yoy_ratio',
    'sr_red_roll3_mean',
    'sr_red_roll3_std',
    'sr_red_roll3_sum',
    'sar_rvi_lag1',
    'sar_rvi_lag2',
    'sar_rvi_lag3',
    'sar_rvi_diff1',
    'sar_rvi_ratio1',
    'sar_rvi_yoy_diff',
    'sar_rvi_yoy_ratio',
    'sar_rvi_roll3_mean',
    'sar_rvi_roll3_std',
    'sar_rvi_roll3_sum',
    'sar_vv_vh_ratio_lag1',
    'sar_vv_vh_ratio_lag2',
    'sar_vv_vh_ratio_lag3',
    'sar_vv_vh_ratio_diff1',
    'sar_vv_vh_ratio_ratio1',
    'sar_vv_vh_ratio_yoy_diff',
    'sar_vv_vh_ratio_yoy_ratio',
    'sar_vv_vh_ratio_roll3_mean',
    'sar_vv_vh_ratio_roll3_std',
    'sar_vv_vh_ratio_roll3_sum',
    's1_vv_lag1',
    's1_vv_lag2',
    's1_vv_lag3',
    's1_vv_diff1',
    's1_vv_ratio1',
    's1_vv_yoy_diff',
    's1_vv_yoy_ratio',
    's1_vv_roll3_mean',
    's1_vv_roll3_std',
    's1_vv_roll3_sum',
    's1_vh_lag1',
    's1_vh_lag2',
    's1_vh_lag3',
    's1_vh_diff1',
    's1_vh_ratio1',
    's1_vh_yoy_diff',
    's1_vh_yoy_ratio',
    's1_vh_roll3_mean',
    's1_vh_roll3_std',
    's1_vh_roll3_sum',
    'et_deficit_lag1',
    'et_deficit_lag2',
    'et_deficit_lag3',
    'et_deficit_diff1',
    'et_deficit_ratio1',
    'et_deficit_yoy_diff',
    'et_deficit_yoy_ratio',
    'et_deficit_roll3_mean',
    'et_deficit_roll3_std',
    'et_deficit_roll3_sum',
    'et_fraction_lag1',
    'et_fraction_lag2',
    'et_fraction_lag3',
    'et_fraction_diff1',
    'et_fraction_ratio1',
    'et_fraction_yoy_diff',
    'et_fraction_yoy_ratio',
    'et_fraction_roll3_mean',
    'et_fraction_roll3_std',
    'et_fraction_roll3_sum',
    'lst_diurnal_range_lag1',
    'lst_diurnal_range_lag2',
    'lst_diurnal_range_lag3',
    'lst_diurnal_range_diff1',
    'lst_diurnal_range_ratio1',
    'lst_diurnal_range_yoy_diff',
    'lst_diurnal_range_yoy_ratio',
    'lst_diurnal_range_roll3_mean',
    'lst_diurnal_range_roll3_std',
    'lst_diurnal_range_roll3_sum',
    'ndvi_anom',
    'ndvi_anom_std',
    'vci',
    'lst_anom',
    'tci',
    'vhi',
    'drought_mild_pc',
    'drought_severe_pc',
    'drought_extreme',
    'compound_stress_pc',
    'vci_lag1',
    'vci_lag2',
    'vci_lag3',
    'vci_roll3_mean',
    'tci_lag1',
    'tci_lag2',
    'tci_lag3',
    'tci_roll3_mean',
    'vhi_lag1',
    'vhi_lag2',
    'vhi_lag3',
    'vhi_roll3_mean',
    'fire_detected',
    'fire_cumulative_count',
    'months_since_fire',
    'fire_frp_cumulative',
    'fire_count_12m',
    'ndvi_lt_mean',
    'ndvi_lt_std',
    'ndvi_lt_p10',
    'ndvi_lt_p25',
    'ndvi_lt_p75',
    'ndvi_lt_p90',
    'ndvi_lt_cv',
    'ndvi_drought_year_count',
    'ndvi_mean_mam',
    'ndvi_mean_ond',
    'ndvi_mean_jfas',
    'm_2',
    'm_3',
    'm_4',
    'm_5',
    'm_6',
    'm_7',
    'm_8',
    'm_9',
    'm_10',
    'm_11',
    'm_12',
]

# XGB parameters
use_gpu = False
param = dict(
    objective="reg:squarederror",
    max_depth=5,
    eta=0.2,
    gamma=4,
    min_child_weight=6,
    subsample=0.7,
    tree_method="gpu_hist" if use_gpu else "hist",  # Use GPU accelerated algorithm
    early_stopping_rounds=10 # stop training if val loss not improving every x rounds
)
num_round = 50

### Merge input data and saved to `processed` data directory

In [22]:
file_name = 'dataset_for_crossval.parquet' # set this to whatever you want to name the file

lst = [LOCATION_JOIN_KEY] + [DATE_JOIN_KEY]
flattened_keys = [item for sublist in lst for item in (sublist if isinstance(sublist, list) else [sublist])]

feat_df = pd.read_parquet(FEAT_DATA_PATH)[flattened_keys + FEAT_LS]
target_df = pd.read_csv(TARGET_DATA_PATH)[[
                                        'hhid', 
                                        'gps_latitude', 
                                        'gps_longitude',
                                        'ibli_date',
                                        'year', 
                                        'month',
                                        'ibli_dekad',
                                        'data_observed',
                                        'season',
                                        LABEL_COLUMN
                                    ]]

merged_df = feat_df.merge(target_df,
                          on=flattened_keys,
                          suffixes=['_feat', '_target'])
merged_df['year_month'] = pd.to_datetime(
    merged_df['year'].astype(str) + merged_df['month'].astype(str).str.zfill(2),
    format='%Y%m').dt.to_period('M')  # gives Period('2008-01', 'M') etc.
merged_df.to_parquet(f's3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/data/training/processed/{file_name}')

## Setup environment

In [23]:
sagemaker_session = sagemaker.session.Session()
role = sagemaker.get_execution_role()
bucket = sagemaker_session.default_bucket()
region = sagemaker_session.boto_region_name

pipeline_name = "lmr-xgb-training-pipeline"
instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.xlarge")

# Mlflow
tracking_server_arn = "arn:aws:sagemaker:us-east-1:575108933641:mlflow-tracking-server/lmr-tracking-server-5t7l23o0xvt99j-chws71x3trpelj-dev"
experiment_name = "lmr-sm-pipelines-experiment"

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


Write `requirements` and `config` files that'll be used by the steps in our SageMaker Pipeline

In [24]:
%%writefile config.yaml
SchemaVersion: '1.0'
SageMaker:
  PythonSDK:
    Modules:
      RemoteFunction:
        # role arn is not required if in SageMaker Notebook instance or SageMaker Studio
        # Uncomment the following line and replace with the right execution role if in a local IDE
        # RoleArn: <replace the role arn here>
        InstanceType: ml.m5.xlarge
        Dependencies: ./requirements.txt
        IncludeLocalWorkDir: true
        CustomFileFilter:
          IgnoreNamePatterns: # files or directories to ignore
          - "*.ipynb" # all notebook files

Overwriting config.yaml


In [25]:
# Set path to config file
os.environ["SAGEMAKER_USER_CONFIG_OVERRIDE"] = os.getcwd()

In [26]:
%%writefile requirements.txt
scikit-learn
xgboost==2.1.4
s3fs==2024.12.0
sagemaker==2.254.1
pandas>=2.0.0
gevent
geventhttpclient
shap
matplotlib
fsspec
mlflow==2.13.2
sagemaker-mlflow==0.1.0
cloudpickle==3.1.2
geopandas
shapely
rasterio

Overwriting requirements.txt


## Define the SageMaker Pipeline

### Preprocessing Step

In [27]:
# Location of our dataset
input_path = f"s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/data/training/processed/{file_name}"

The breast cancer Wisconsin dataset contains column `id` which we do not use for training. The second column `diagnosis` is class label, and the label is represented using 'M' for Malignant class, and 'B' for Benign class. 

In the preprocessing step, we drop the column `id`, then split the dataset into three distinct sets: train, validation, and test set.

Note that `keep_alive_period_in_seconds` parameter in @step decorator indicates how many seconds we want to keep the instance alive, waiting to be reused for the next pipeline step execution. Setting this parameter speeds up the pipeline execution because we reduce the launching of new instances to execute pipeline steps.

In [28]:
@step(
    name="DataPreprocessing",
    instance_type=instance_type,
)
def preprocess(
    raw_data_s3_path,
    output_prefix,
    output_s3_base_uri,
    experiment_name,
    run_name,
    label_column,
    date_column,
    min_train_years,
    step_years,
    feature_names,
):
    from preprocess import run_preprocess
    # Returns: fold_paths[0], test_s3_path[1], test_metadata_s3_path[2], experiment_name[3], run_id[4]
    return run_preprocess(
        raw_data_s3_path=raw_data_s3_path,
        output_prefix=output_prefix,
        output_s3_base_uri=output_s3_base_uri,
        experiment_name=experiment_name,
        run_name=run_name,
        label_column=label_column,
        date_column=date_column,
        min_train_years=min_train_years,
        step_years=step_years,
        feature_names=feature_names,
    )

### Training Step

We train a XGBoost model in this training step, using @step-decorated function with the S3 path of training and validation set, along with XGBoost hyperparameters. The S3 paths for both training and validation set is coming from the output of the previous step.

This could be replaced or supplemented with an option for selecting a different model architecture

In [29]:
@step(
    name="ModelTraining",
    instance_type=instance_type,
)
def train(
    fold_paths: list,           # list of dicts from preprocess step
    experiment_name: str,
    run_id: str,
    tracking_server_arn: str = tracking_server_arn,
    param: dict = param,
    num_round: int = num_round,
    label_column: str = "tlu_loss_ratio",
):
    """
    Walk-forward CV training.
 
    For each fold:
      - Train an XGBoost model on fold['train']
      - Validate on fold['val']
      - Log fold metrics as a nested MLflow run
 
    After all folds, train a FINAL model on the union of all training data
    from the last fold (i.e., all data up to but not including the test set).
    This final model is what gets logged as the registered artifact.
 
    Returns
    -------
    experiment_name, run_id, training_run_id, top_features, cv_metrics_summary
    """
    import json
    import tempfile
    import numpy as np
    import pandas as pd
    import mlflow
    from xgboost import XGBRegressor
    from sklearn.metrics import (
        mean_squared_error, mean_absolute_error,
        mean_absolute_percentage_error, r2_score,
    )
 
    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)
 
    with mlflow.start_run(run_id=run_id):
        mlflow.xgboost.autolog(
            log_input_examples=False,   # skip per-fold to avoid noise
            log_model_signatures=True,
            log_models=False,           # we log the final model manually
            log_datasets=False,
            model_format="xgb",
        )
 
        # ── Per-fold CV ───────────────────────────────────────────────────────
        fold_metrics_list = []
 
        for fold in fold_paths:
            fold_idx = fold["fold_index"]
            val_years_str = ",".join([str(y) for y in fold["val_years"]])
            with mlflow.start_run(
                run_name=f"CV_Fold_{fold_idx}_val_{val_years_str}",
                nested=True,
            ):
                train_df = pd.read_csv(fold["train"])
                val_df   = pd.read_csv(fold["val"])
 
                y_train = train_df.pop(label_column).astype(float)
                y_val   = val_df.pop(label_column).astype(float)
 
                model = XGBRegressor(n_estimators=num_round, **param)
                model.fit(
                    train_df, y_train,
                    eval_set=[(val_df, y_val)],
                    verbose=False,
                )
 
                y_pred = model.predict(val_df)
                fold_rmse = float(np.sqrt(mean_squared_error(y_val, y_pred)))
                fold_mae  = float(mean_absolute_error(y_val, y_pred))
                fold_r2   = float(r2_score(y_val, y_pred))
 
                mlflow.log_params({
                    "fold_index":   fold_idx,
                    "val_years":  ",".join([str(y) for y in fold["val_years"]]),
                    "train_rows":   fold["train_rows"],
                    "val_rows":     fold["val_rows"],
                })
                mlflow.log_metrics({
                    "fold_rmse": fold_rmse,
                    "fold_mae":  fold_mae,
                    "fold_r2":   fold_r2,
                })
 
                fold_metrics_list.append({
                    "fold_index": fold_idx,
                    "val_year": ",".join([str(y) for y in fold["val_years"]]),
                    "rmse": fold_rmse,
                    "mae":  fold_mae,
                    "r2":   fold_r2,
                })
                print(
                    f"Fold {fold_idx} (val={val_years_str}): "
                    f"RMSE={fold_rmse:.4f}  MAE={fold_mae:.4f}  R²={fold_r2:.4f}"
                )
 
        # ── Summarise CV results ──────────────────────────────────────────────
        cv_metrics_summary = {
            "cv_mean_rmse": float(np.mean([m["rmse"] for m in fold_metrics_list])),
            "cv_std_rmse":  float(np.std( [m["rmse"] for m in fold_metrics_list])),
            "cv_mean_mae":  float(np.mean([m["mae"]  for m in fold_metrics_list])),
            "cv_mean_r2":   float(np.mean([m["r2"]   for m in fold_metrics_list])),
            "n_folds":      len(fold_metrics_list),
        }
        mlflow.log_metrics(cv_metrics_summary)
        print(f"CV summary: {cv_metrics_summary}")

        # ── Print fold-by-fold performance table ─────────────────────────────────
        import pandas as pd
        fold_df = pd.DataFrame(fold_metrics_list).set_index("fold_index")
        fold_df = fold_df[["val_year", "rmse", "mae", "r2"]]
        fold_df.loc["MEAN"] = ["—", cv_metrics_summary["cv_mean_rmse"],
                                    cv_metrics_summary["cv_mean_mae"],
                                    cv_metrics_summary["cv_mean_r2"]]
        fold_df.loc["STD"]  = ["—", cv_metrics_summary["cv_std_rmse"], "—", "—"]
        print("\n── Cross-Validation Performance by Fold ──")
        print(fold_df.to_string())
        
        # Also log the full table as an MLflow artifact so it's retrievable later
        with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False) as tmp:
            fold_df.to_csv(tmp)
            tmp_path = tmp.name
        mlflow.log_artifact(tmp_path, artifact_path="cv_fold_metrics")
 
        # ── Final model: retrain on ALL training data (last fold's full train set)
        # The last fold has the largest training window — everything up to the
        # held-out test period.
        # ─────────────────────────────────────────────────────────────────────
        with mlflow.start_run(run_name="FinalModel", nested=True) as final_run:
            training_run_id = final_run.info.run_id
 
            last_fold = fold_paths[-1]
            train_df  = pd.read_csv(last_fold["train"])
            # Also append the val rows from the last fold — they are no longer
            # needed for validation once CV is complete, and more data is better.
            val_df    = pd.read_csv(last_fold["val"])
            full_train_df = pd.concat([train_df, val_df], ignore_index=True)
 
            y_full = full_train_df.pop(label_column).astype(float)
 
            import copy
            final_params = copy.deepcopy(param)
            final_params.pop("early_stopping_rounds", None)
            
            final_model = XGBRegressor(n_estimators=num_round, **final_params)
            final_model.fit(full_train_df, y_full)
 
            mlflow.sklearn.log_model(
                final_model,
                artifact_path="model",
                input_example=full_train_df[:5],
            )
            mlflow.log_params({"final_train_rows": len(full_train_df)})
 
            # ── Top-3 features by built-in importance ─────────────────────────
            try:
                scores = final_model.get_booster().get_fscore()
                top_features = sorted(scores, key=scores.get, reverse=True)[:3]
            except Exception as e:
                print(f"Could not extract feature importance: {e}")
                top_features = list(full_train_df.columns[:3])
 
            print(f"Top-3 features: {top_features}")
            mlflow.log_param("top_features", top_features)
 
    return experiment_name, run_id, training_run_id, top_features, cv_metrics_summary

### Custom Pyfunc Model Wrapper

Wraps the trained XGBoost regressor so that every call to `.predict()` returns a structured dict containing the raw prediction, a bucketed category, and the top-3 most important SHAP features.

In [30]:
# NOTE: This class must be defined in the notebook so it is serialised into
# the local workspace directory that SageMaker ships to each remote step.
# The @step decorator includes IncludeLocalWorkDir=true, so this class is
# available at runtime on the remote instance without any extra packaging.

import mlflow
class TluLossPyfuncModel(mlflow.pyfunc.PythonModel):
    """
    Custom MLflow pyfunc that enriches raw XGBoost predictions with:
      - prediction   : raw float output of the regressor
      - bucket       : 'low' (<0.2) | 'medium' (0.2–0.5) | 'high' (>=0.5)
                       TODO: replace thresholds with % above/below average
      - top_features : list of the 3 features with highest mean |SHAP| value
                       (computed once at training time and frozen into the model)
    """

    # Bucket thresholds — placeholder values, replace with domain cutoffs
    LOW_THRESHOLD    = 0.2  # prediction < LOW_THRESHOLD  → 'low'
    MEDIUM_THRESHOLD = 0.5  # prediction < MEDIUM_THRESHOLD → 'medium', else 'high'

    def __init__(self, xgb_model, top_features: list):
        self.xgb_model   = xgb_model      # unwrapped XGBRegressor
        self.top_features = top_features  # list[str], frozen at training time

    @staticmethod
    def _bucket(value: float) -> str:
        """Map a scalar prediction to a category label."""
        if value < TluLossPyfuncModel.LOW_THRESHOLD:
            return "low"
        elif value < TluLossPyfuncModel.MEDIUM_THRESHOLD:
            return "medium"
        return "high"

    def predict(self, context, model_input):
        """
        Parameters
        ----------
        model_input : pd.DataFrame
            Feature matrix with the same columns used during training.

        Returns
        -------
        list[dict]
            One dict per row, each with keys:
              'prediction' (float), 'bucket' (str), 'top_features' (list[str])
        """
        import numpy as np
        raw_preds = np.array(self.xgb_model.predict(model_input)).flatten()
        return [
            {
                "prediction":   float(pred),
                "bucket":       self._bucket(float(pred)),
                "top_features": self.top_features,
            }
            for pred in raw_preds
        ]


/opt/conda/lib/python3.11/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


### Evaluation Step

In this step, we create a @step-decorated function to evaluate the trained XGBoost model on the test dataset.

In [31]:
@step(name="ModelEvaluation", instance_type=instance_type)
def evaluate(test_s3_path, experiment_name, run_id, training_run_id, label_column='tlu_loss') -> dict:
    import mlflow
    import pandas as pd
    import numpy as np
    from sklearn.metrics import (
        mean_squared_error,
        mean_absolute_error,
        mean_absolute_percentage_error,
        r2_score,
    )

    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_id=run_id):
        with mlflow.start_run(run_name="ModelEvaluation", nested=True):

            test_df = pd.read_csv(test_s3_path)
            y_true = test_df.pop(label_column).astype(float)
            X_test = test_df

            # Debug: confirm data loaded correctly
            print(f"Test set shape: {X_test.shape}")
            print(f"y_true sample: {y_true[:5].tolist()}")
            print(f"y_true range: {y_true.min():.3f} to {y_true.max():.3f}")

            model = mlflow.pyfunc.load_model(f"runs:/{training_run_id}/model")
            y_pred = model.predict(X_test)
            y_pred = np.array(y_pred).flatten().astype(float)

            print(f"y_pred sample: {y_pred[:5].tolist()}")
            print(f"y_pred range: {y_pred.min():.3f} to {y_pred.max():.3f}")

            # Core metrics
            rmse = np.sqrt(mean_squared_error(y_true, y_pred))
            mae  = mean_absolute_error(y_true, y_pred)
            r2   = r2_score(y_true, y_pred)

            # Normalized metrics (useful when label scale varies)
            label_range = y_true.max() - y_true.min()
            nrmse = rmse / label_range if label_range > 0 else float("nan")  # 0–1 scale
            mape  = mean_absolute_percentage_error(y_true, y_pred)           # % error

            # Naive baseline comparison
            naive_pred  = np.full_like(y_true, y_true.mean())
            naive_rmse  = np.sqrt(mean_squared_error(y_true, naive_pred))
            skill_score = 1 - (rmse / naive_rmse)  # >0 means better than naive mean

            metrics = {
                "rmse":        rmse,
                "mae":         mae,
                "r2_score":    r2,
                "nrmse":       nrmse,       # normalized RMSE
                "mape":        mape,        # mean absolute % error
                "naive_rmse":  naive_rmse,  # baseline to beat
                "skill_score": skill_score, # how much better than naive (>0 is good)
            }

            print(f"Metrics: {metrics}")
            mlflow.log_metrics(metrics)  # log all at once, inside the nested run

            return metrics

### Model Registration

In this step, we create a @step-decorated function to register our XGBoost model. 

In [32]:
@step(
    name="ModelRegistration",
    instance_type=instance_type,
)
def register(
    pipeline_name: str,
    experiment_name: str,
    run_id: str,
    training_run_id: str,
    top_features: list,
):
    import json
    import tempfile
    import mlflow

    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_id=run_id):
        with mlflow.start_run(run_name="ModelRegistration", nested=True):

            # Load the raw XGBRegressor that was logged in ModelTraining
            raw_model = mlflow.sklearn.load_model(f"runs:/{training_run_id}/model")

            # Build the enriched pyfunc wrapper, baking in the top-3 features
            enriched_model = TluLossPyfuncModel(
                xgb_model=raw_model,
                top_features=top_features,
            )

            # Log the pyfunc wrapper as the registered artefact
            # This replaces the raw sklearn model as the canonical endpoint
            model_info = mlflow.pyfunc.log_model(
                artifact_path="enriched_model",
                python_model=enriched_model,
            )

            # Also log the inference schema as a JSON artifact for reference
            schema = {
                "output_fields": {
                    "prediction":   "float — raw tlu_loss regression output",
                    "bucket":       "str  — 'low' (<0.2) | 'medium' (0.2–0.5) | 'high' (>=0.5)  [placeholder thresholds]",
                    "top_features": "list[str] — top-3 features by mean absolute SHAP value",
                },
                "top_features_frozen_at_training": top_features,
            }
            with tempfile.NamedTemporaryFile(
                mode="w", suffix=".json", delete=False
            ) as tmp:
                json.dump(schema, tmp, indent=2)
                tmp_path = tmp.name
            mlflow.log_artifact(tmp_path, artifact_path="inference_schema")
            print(f"Logged inference schema: {schema}")

            # Register the pyfunc model in the MLflow Model Registry
            mlflow.register_model(model_info.model_uri, pipeline_name)


### PostProcessing Step

Aggregates household-level predictions to ADMIN3 (ward) level using spatial boundaries.
Outputs risk levels, confidence scores, and top SHAP features per ward.

In [33]:
# S3 path to Kenya ADMIN3 ward boundary shapefile
ADMIN3_SHAPEFILE_S3_PATH = (
    "s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/data/kenya_wards/Kenya wards.shp"
)

@step(name="PostProcessing", instance_type=instance_type)
def postprocess_step(
    test_s3_path: str,
    test_metadata_s3_path: str,
    experiment_name: str,
    run_id: str,
    training_run_id: str,
    admin3_shapefile_path: str,
    label_column: str = "tlu_loss_ratio",
    feature_names: list = None,
) -> dict:
    import mlflow
    import pandas as pd
    import numpy as np

    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)

    # 1. Generate predictions on the test set
    with mlflow.start_run(run_id=run_id):
        with mlflow.start_run(run_name="GeneratePredictions", nested=True):
            test_df = pd.read_csv(test_s3_path)
            y_true = test_df.pop(label_column).astype(float)
            X_test = test_df.copy()
            model = mlflow.pyfunc.load_model(f"runs:/{training_run_id}/model")
            y_pred = np.array(model.predict(X_test)).flatten().astype(float)

            # Rebuild a predictions CSV with coordinates + features + prediction
            # Load test data with GPS metadata for spatial join
            raw_test = pd.read_csv(test_metadata_s3_path)
            raw_test["prediction"] = y_pred

            # Write predictions to S3
            pred_path = test_s3_path.rsplit("/", 1)[0] + "/predictions_with_coords.csv"
            raw_test.to_csv(pred_path, index=False)
            print(f"Wrote {len(raw_test)} predictions to {pred_path}")

    # 2. Run ADMIN3 ward-level aggregation
    from postprocess import run_postprocess
    ward_csv, ward_geojson, ward_geotiff = run_postprocess(
        predictions_s3_path=pred_path,
        experiment_name=experiment_name,
        run_id=run_id,
        training_run_id=training_run_id,
        admin3_shapefile_path=admin3_shapefile_path,
        label_column=label_column,
        feature_names=feature_names,
    )
    return {"ward_csv": ward_csv, "ward_geojson": ward_geojson, "ward_geotiff": ward_geotiff}

## Creating the SageMaker Pipeline

We connect all defined pipeline `@step` functions into a multi-step pipeline. Then, we submit and execute the pipeline.

In [34]:
output_s3_base_uri = "s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/pipeline-artifacts"

preprocessing_step = preprocess(
    raw_data_s3_path=input_path,
    output_prefix=f"{pipeline_name}/dataset/v2",
    output_s3_base_uri=output_s3_base_uri,
    experiment_name=experiment_name,
    run_name=ExecutionVariables.PIPELINE_EXECUTION_ID,
    label_column=LABEL_COLUMN,
    date_column='year_month',
    min_train_years=1,
    step_years=1,
    feature_names=FEAT_LS,
)
# Returns: fold_paths[0], test_s3_path[1], test_metadata_s3_path[2], experiment_name[3], run_id[4]

training_step = train(
    fold_paths=preprocessing_step[0],
    experiment_name=preprocessing_step[3],
    run_id=preprocessing_step[4],
    label_column=LABEL_COLUMN,
    tracking_server_arn=tracking_server_arn,
)
# Returns: experiment_name[0], run_id[1], training_run_id[2], top_features[3], cv_metrics_summary[4]

eval_step = evaluate(
    test_s3_path=preprocessing_step[1],
    experiment_name=preprocessing_step[3],
    run_id=preprocessing_step[4],
    training_run_id=training_step[2],
    label_column=LABEL_COLUMN,
)

# PostProcessing: aggregate predictions to ADMIN3 ward level
postprocessing_step = postprocess_step(
    test_s3_path=preprocessing_step[1],
    test_metadata_s3_path=preprocessing_step[2],
    experiment_name=preprocessing_step[3],
    run_id=preprocessing_step[4],
    training_run_id=training_step[2],
    admin3_shapefile_path=ADMIN3_SHAPEFILE_S3_PATH,
    label_column=LABEL_COLUMN,
    feature_names=FEAT_LS,
)

conditional_register_step = ConditionStep(
    name="ConditionalRegister",
    conditions=[
        ConditionGreaterThanOrEqualTo(
            left=eval_step["r2_score"],
            right=0,  # only register if R² >= 0 on held-out test set
        )
    ],
    if_steps=[
        register(
            pipeline_name=pipeline_name,
            experiment_name=preprocessing_step[3],
            run_id=preprocessing_step[4],
            training_run_id=training_step[2],
            top_features=training_step[3],
        )
    ],
    else_steps=[FailStep(name="Fail", error_message="Model performance is not good enough")],
)

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[instance_type],
    steps=[preprocessing_step, training_step, postprocessing_step, conditional_register_step],
)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


In [35]:
pipeline.upsert(role_arn=role)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.Dependencies
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.IncludeLocalWorkDir
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.CustomFileFilter.IgnoreNamePatterns
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.SecurityGroupIds


2026-04-01 21:59:12,581 sagemaker.remote_function INFO     Uploading serialized function code to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/DataPreprocessing/2026-04-01-21-59-10-351/function
2026-04-01 21:59:12,660 sagemaker.remote_function INFO     Uploading serialized function arguments to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/DataPreprocessing/2026-04-01-21-59-10-351/arguments
2026-04-01 21:59:13,001 sagemaker.remote_function INFO     Copied dependencies file at './requirements.txt' to '/tmp/tmpjkqce90a/requirements.txt'
2026-04-01 21:59:13,044 sagemaker.remote_function INFO     Successfully uploaded dependencies and pre execution scripts to 's3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/DataPreprocessing/2026-04-01-21-59-10-351/pre_exec_script_and

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.Dependencies
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.IncludeLocalWorkDir
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.CustomFileFilter.IgnoreNamePatterns
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.SecurityGroupIds


2026-04-01 21:59:14,978 sagemaker.remote_function INFO     Uploading serialized function code to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelTraining/2026-04-01-21-59-10-351/function
2026-04-01 21:59:15,051 sagemaker.remote_function INFO     Uploading serialized function arguments to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelTraining/2026-04-01-21-59-10-351/arguments
2026-04-01 21:59:15,191 sagemaker.remote_function INFO     Copied dependencies file at './requirements.txt' to '/tmp/tmppwav3e1l/requirements.txt'
2026-04-01 21:59:15,230 sagemaker.remote_function INFO     Successfully uploaded dependencies and pre execution scripts to 's3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelTraining/2026-04-01-21-59-10-351/pre_exec_script_and_dependencie

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.Dependencies
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.IncludeLocalWorkDir
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.CustomFileFilter.IgnoreNamePatterns
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.SecurityGroupIds


...

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.Dependencies
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.IncludeLocalWorkDir
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.CustomFileFilter.IgnoreNamePatterns
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.SecurityGroupIds


2026-04-01 21:59:19,149 sagemaker.remote_function INFO     Uploading serialized function code to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelRegistration/2026-04-01-21-59-10-351/function
2026-04-01 21:59:19,227 sagemaker.remote_function INFO     Uploading serialized function arguments to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelRegistration/2026-04-01-21-59-10-351/arguments
2026-04-01 21:59:19,362 sagemaker.remote_function INFO     Copied dependencies file at './requirements.txt' to '/tmp/tmpqtdtrwb7/requirements.txt'
2026-04-01 21:59:19,401 sagemaker.remote_function INFO     Successfully uploaded dependencies and pre execution scripts to 's3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelRegistration/2026-04-01-21-59-10-351/pre_exec_script_and

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.Dependencies
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.IncludeLocalWorkDir
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.CustomFileFilter.IgnoreNamePatterns
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.SecurityGroupIds


2026-04-01 21:59:21,277 sagemaker.remote_function INFO     Uploading serialized function code to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelEvaluation/2026-04-01-21-59-10-351/function
2026-04-01 21:59:21,446 sagemaker.remote_function INFO     Uploading serialized function arguments to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelEvaluation/2026-04-01-21-59-10-351/arguments
2026-04-01 21:59:21,582 sagemaker.remote_function INFO     Copied dependencies file at './requirements.txt' to '/tmp/tmp_nlfagr1/requirements.txt'
2026-04-01 21:59:21,684 sagemaker.remote_function INFO     Successfully uploaded dependencies and pre execution scripts to 's3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelEvaluation/2026-04-01-21-59-10-351/pre_exec_script_and_depen

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:575108933641:pipeline/lmr-xgb-training-pipeline',
 'PipelineVersionId': 72,
 'ResponseMetadata': {'RequestId': 'df17985a-ab3c-44fc-805b-13429e37de0f',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'df17985a-ab3c-44fc-805b-13429e37de0f',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '116',
   'date': 'Wed, 01 Apr 2026 21:59:24 GMT'},
  'RetryAttempts': 0}}

## Execute the SageMaker Pipeline

In [36]:
execution = pipeline.start()

In [37]:
execution.describe()

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:575108933641:pipeline/lmr-xgb-training-pipeline',
 'PipelineExecutionArn': 'arn:aws:sagemaker:us-east-1:575108933641:pipeline/lmr-xgb-training-pipeline/execution/g5wqfykgapq7',
 'PipelineExecutionDisplayName': 'execution-1775080764415',
 'PipelineExecutionStatus': 'Executing',
 'CreationTime': datetime.datetime(2026, 4, 1, 21, 59, 24, 303000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2026, 4, 1, 21, 59, 24, 303000, tzinfo=tzlocal()),
 'CreatedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:575108933641:user-profile/d-lf2eivtg2b7i/c4083418-d061-7068-829e-604008d36196',
  'UserProfileName': 'c4083418-d061-7068-829e-604008d36196',
  'DomainId': 'd-lf2eivtg2b7i',
  'IamIdentity': {'Arn': 'arn:aws:sts::575108933641:assumed-role/datazone_usr_role_5t7l23o0xvt99j_5w5uuggwdfhu3r/SageMaker',
   'PrincipalId': 'AROAYLZZJ5AEZSW7N2N4C:SageMaker',
   'SourceIdentity': 'c4083418-d061-7068-829e-604008d36196'}},
 'LastModifiedBy': {'User

In [38]:
execution.wait()

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 execution.wait()                                                                             │
│   2                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/sagemaker/workflow/pipeline.py:989 in wait               │
│                                                                                                  │
│    986 │   │   waiter = botocore.waiter.create_waiter_with_client(                               │
│    987 │   │   │   waiter_id, model, self.sagemaker_session.sagemaker_client                     │
│    988 │   │   )                                                                                 │
│ ❱  989 │   │   waiter.wait(PipelineExecutionArn=self.arn)                                        │
│    990 │                                                                                         │
│    991 │   def result(self, step_name: str):                                                     │
│    992 │   │   """Retrieves the output of the provided step if it is a ``@step`` decorated func  │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/botocore/waiter.py:58 in wait                            │
│                                                                                                  │
│    55 │   # Waiter.wait method. This is needed to attach a docstring to the                      │
│    56 │   # method.                                                                              │
│    57 │   def wait(self, **kwargs):                                                              │
│ ❱  58 │   │   Waiter.wait(self, **kwargs)                                                        │
│    59 │                                                                                          │
│    60 │   wait.__doc__ = WaiterDocstring(                                                        │
│    61 │   │   waiter_name=waiter_name,                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/botocore/context.py:123 in wrapper                       │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/botocore/waiter.py:378 in wait                           │
│                                                                                                  │
│   375 │   │   │   │   return                                                                     │
│   376 │   │   │   if current_state == 'failure':           

In [ ]:
execution.list_steps()